# 使用 PROC NESTED 分解保险公司组织层级中的理赔结算变异性

## 内容摘要

一家财产与意外险保险公司想知道理赔结算中的不一致性究竟**源自何处**：是地区之间的差异、分公司之间的差异、理算员个体之间的差异，还是仅仅是逐笔理赔的噪声？本笔记本构建了一套合成的、完全嵌套的车险理赔数据（区域 > 分公司 > 理算员 > 理赔），并使用 **PROC NESTED** 执行随机效应方差分析，估计层级结构中每一层所贡献的方差分量，并以占总方差百分比的形式报告。结果可以告诉管理层应针对哪个组织层级来改善结算一致性。

## 数据来源

所有数据均由第一个 DATA 步骤内联生成——不使用任何外部文件，也不联网。设计是完全嵌套的：每个分公司恰好属于一个区域，每位理算员恰好属于一个分公司，每笔理赔恰好由一位理算员处理。

| 数据集 | 行数 | 变量 | 类型 | 角色 | 说明 |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | 字符型 | CLASS（第 1 层，最外层） | 地理区域（5 个区域：NE、SE、MW、WC、SW） |
| | | `Branch` | 字符型 | CLASS（第 2 层，嵌套于 Region） | 分公司代码（每个区域 2 家分公司） |
| | | `Adjuster` | 字符型 | CLASS（第 3 层，嵌套于 Branch） | 理算员 ID（每家分公司 2 位理算员） |
| | | `Settlement` | 数值型 | VAR（因变量） | 车险理赔的赔付金额，单位为美元 |
| | | `CycleDays` | 数值型 | VAR（因变量） | 从首次出险通知到结案的天数 |

合成结构：5 个区域 &times; 2 家分公司 &times; 2 位理算员 &times; 2 笔理赔 = 40 个观测值。通过 `rand('NORMAL', ...)` 以加法方式叠加区域随机效应、分公司（嵌套于区域）随机效应、理算员（嵌套于分公司）随机效应以及理赔层面的残差，使各方差分量可被恢复。区域效应采用最大的标准差（2,200）生成，因此*理赔在哪个区域处理*是主导因素。种子通过 `call streaminit(20260531)` 固定。这种紧凑且完全平衡的设计使层级结构的每一层都拥有真实的自由度。

# 使用 PROC NESTED 分解理赔结算变异性

当一家财产与意外险保险公司审视其车险理赔的结算金额时，往往会发现很大的差异。其中一些差异是不可避免的（每一起事故都不相同），但也有一部分反映了**组织**因素——某个区域整体赔付更高、某家分公司更慷慨、某位理算员是异常值。

这些数据具有严格的**嵌套（层级）**结构：

```
区域（Region） -> 分公司（Branch，嵌套于区域） -> 理算员（Adjuster，嵌套于分公司） -> 单笔理赔
```

一家分公司恰好属于一个区域，一位理算员恰好属于一家分公司，因此这些因素是嵌套的，而非交叉的。`PROC NESTED` 正是为这种设计执行随机效应方差分析，并估计每一层的**方差分量**，以占总方差的百分比表示。这个百分比拆分回答了业务问题：*应针对组织的哪一层，才能让结算结果更加一致？*

## 步骤 1 &mdash; 生成一套合成的、完全嵌套的理赔数据

我们模拟 5 个区域，每个区域 2 家分公司，每家分公司 2 位理算员，每位理算员处理 2 笔理赔（共 40 行）。我们通过加法方式叠加随机效应来构建响应变量，使每一层都真正贡献方差：

- **区域**效应（离散程度最大&mdash;&mdash;区域间差异最明显），
- **分公司（嵌套于区域）**效应，
- **理算员（嵌套于分公司）**效应，
- 以及**理赔层面**的残差（理算员内部的噪声）。

`call streaminit` 固定种子以保证可重现性；每个效应均通过 `rand('NORMAL', mean, sd)` 生成。区域效应使用最大的标准差，因此理应占总方差的最大份额。我们还生成了第二个响应变量 `CycleDays`，它共享相同的层级结构，以便稍后演示多响应分析。

In [1]:
数据 claims;
   调用 streaminit(20260531);
   长度 Region $2 Branch $6 Adjuster $9;

   /* 区域层面的随机效应：区域间差异最大 */
   循环 r = 1 到 5;
      Region = scan('NE SE MW WC SW', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* 分公司嵌套于区域之内 */
      循环 b = 1 到 2;
         Branch = catx('-', Region, put(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* 理算员嵌套于分公司之内 */
         循环 a = 1 到 2;
            Adjuster = catx('-', Branch, put(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* 该理算员处理的单笔理赔 */
            循环 claim = 1 到 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               如果 Settlement < 0 那么 Settlement = 0;
               如果 CycleDays  < 1 那么 CycleDays  = 1;
               输出;
            结束;
         结束;
      结束;
   结束;

   保留 Region Branch Adjuster Settlement CycleDays;
运行;


NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## 步骤 2 &mdash; 按嵌套层级排序

`PROC NESTED` 要求输入数据按 CLASS 变量列出的顺序排序&mdash;&mdash;最外层因子在前。我们按 `Region Branch Adjuster` 排序，以便该过程能正确遍历层级结构。

In [2]:
过程 sort 数据=claims;
   按照 Region Branch Adjuster;
运行;


NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## 步骤 3 &mdash; 赔付金额的方差分量分析

这是核心分析。我们按从最外层到最内层的顺序列出 CLASS 变量（`Region Branch Adjuster`）；最内层的重复&mdash;&mdash;单笔理赔&mdash;&mdash;会自动作为误差项处理。`VAR Settlement` 指定唯一的响应变量。

`LABEL` 语句为输出标题中的响应变量提供了更友好的显示名称。输出内容包括期望均方系数、方差分析表（含每个方差分量对零的 F 检验），以及每一层的**方差分量**估计值及其**占总方差的百分比**。

In [3]:
标题 '车险理赔结算金额的方差分量';
title2 '区域／分公司／理算员随机效应方差分析';

过程 nested 数据=claims;
   分类 Region Branch Adjuster;
   变量 Settlement;
   label Settlement = '赔付金额 (USD)';
运行;

                                                     车险理赔结算金额的方差分量                                                      
                                                   区域／分公司／理算员随机效应方差分析                                                   


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable 赔付金额 (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826901         


NOTE: Option TITLE changed to 车险理赔结算金额的方差分量.
NOTE: Option TITLE2 changed to 区域／分公司／理算员随机效应方差分析.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## 步骤 4 &mdash; 同时分析两个响应变量

管理层同样关心**结案周期**&mdash;&mdash;理赔结案所需的时间。我们将 `CycleDays` 加入 `VAR` 列表。当存在一个以上的因变量时，`PROC NESTED` 会额外报告一份**协变分析**，展示两个响应变量在层级结构每一层上如何共同变化（例如，赔付更高的区域是否结案也更慢）。

In [4]:
标题 '结算金额与结案周期的方差分量';
title2 '同一嵌套层级下的两个响应变量';

过程 nested 数据=claims;
   分类 Region Branch Adjuster;
   变量 Settlement CycleDays;
   label Settlement = '赔付金额 (USD)'
         CycleDays  = '结案天数';
运行;

                                                     结算金额与结案周期的方差分量                                                     
                                                     同一嵌套层级下的两个响应变量                                                     


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable 赔付金额 (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826901         


NOTE: Option TITLE changed to 结算金额与结案周期的方差分量.
NOTE: Option TITLE2 changed to 同一嵌套层级下的两个响应变量.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## 步骤 5 &mdash; 使用 AOV 选项查看纯方差分量视图

若只需快速查看两个响应变量的方差分量摘要，`AOV` 选项会将输出限制为方差分析统计量，并抑制协变分析部分。这是投资组合分析师在只需要各响应变量的分层方差拆分、而不需要跨响应协变信息时会浏览的精简视图。

In [5]:
标题 '仅方差分量视图 (AOV)';

过程 nested 数据=claims aov;
   分类 Region Branch Adjuster;
   变量 Settlement CycleDays;
   label Settlement = '赔付金额 (USD)'
         CycleDays  = '结案天数';
运行;

标题;

                                                     仅方差分量视图 (AOV)                                                      
                                                     同一嵌套层级下的两个响应变量                                                     


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable 赔付金额 (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826901         


NOTE: Option TITLE changed to 仅方差分量视图 (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## 结果解读

方差分析表中的**占总方差百分比**列是最重要的信息。自上而下阅读该列，可以得到每个组织层级对总结算变异性的贡献占比。对于赔付金额，本次运行结果为：

- **区域（Region）&mdash; 54.4%**（Pr > F = 0.0304）。数据生成时区域层面的离散程度设定为最大，分析结果也印证了这一点：超过一半的结算变异性来自**区域之间**，且具有统计显著性，证明*理赔在哪个区域处理*是主导因素。
- **分公司（嵌套于区域）&mdash; 9.0%**（Pr > F = 0.18）。从区域细分到具体分公司后，还有一小部分额外差异，但此处不显著。
- **理算员（嵌套于分公司）&mdash; 3.7%**（Pr > F = 0.33）。在这批业务中，可归因于理算员个体的差异很小。
- **误差 &mdash; 32.9%。** 这是理算员内部逐笔理赔的残余噪声，是任何组织手段都无法消除的不可约变异。

每一层还附带一个**F 检验（Pr > F）**，用于检验该层方差分量为零的原假设。区域层面较小的 p 值（0.0304）是区域间存在真实系统性差异的统计证据，而非抽样偶然性。

结案周期的响应变量讲述了类似的故事：**区域**解释了结案天数中 **45.8%** 的变异（Pr > F = 0.0448，同样显著），分公司和理算员各自贡献个位数百分比，残差占 40.1%。因此，保险公司赔多少钱、结案要多久，首先都是由区域决定的。

双响应变量的分析还增加了一份**协变分析**。在这里，区域层级主导了交叉乘积，总体**相关系数为 -0.36**&mdash;&mdash;呈负相关关系：赔付金额较高的区域，结案往往*更快*，而不是更慢。这是一个有用且并不直观的发现&mdash;&mdash;赔付高的区域并不是结案慢的区域。

**业务启示：** 由于区域在两个响应变量的占比拆分中都占主导地位，保险公司应首先在**跨区域**范围内统一结算指引和授权限额&mdash;&mdash;这正是最大且具有统计显著性的不一致来源&mdash;&mdash;然后再投入分公司或理算员层面的干预措施。赔付金额与结案周期之间的负相关意味着不存在同时"赔得最多、结案最慢"的单一区域，因此这两个问题可以通过各自独立的、针对区域的方案来解决。`PROC NESTED` 把"我们的结算结果不一致"这种模糊的直觉，转化为逐层、可执行的不一致来源归因。